# 01 — Tổng quan dữ liệu & kết quả (notebook demo)

Notebook này **chỉ đọc artifact** trong `reports/results/*.json` (và gọi lại đúng hàm của
`forecasting/` / `scripts/`) — không viết lại logic, nên số liệu không thể lệch với báo cáo.

Chạy: mở notebook bằng VS Code (Python ≥ 3.11) sau khi đã chạy `python -m scripts.run_all`.

Nội dung: (1) quy mô corpus + phân bố nhãn, (2) nhãn theo công ty, (3) phân bố lớp theo tập,
(4) bảng so sánh mô hình/baseline, (5) mô hình được chốt + ngưỡng, (6) mẫu sai, (7) demo dự đoán.

In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
if (ROOT / 'forecasting').exists() is False:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

RESULTS = ROOT / 'reports' / 'results'

def load(name):
    return json.loads((RESULTS / name).read_text(encoding='utf-8'))

eda = load('eda_summary.json')
corpus = eda['corpus']
print(f"Corpus: {corpus['n_companies']} công ty, {corpus['n_quarters']} quý, {corpus['n_samples']} mẫu")
print('Số feature:', load('summary.json')['n_features'])

## 2. Phân bố nhãn theo CÔNG TY (mất cân bằng thật nằm ở đây)

In [ ]:
share = eda['label']['label_share_by_ticker']
for ticker, value in sorted(share.items(), key=lambda kv: -kv[1]):
    bar = '#' * round(value * 30)
    print(f'{ticker:>5}: {value:5.3f} {bar}')
print('Công ty 100% nhãn 1:', eda['label']['companies_all_one'])
positive = sum(v > 0 for v in share.values())
wa = max(share.values()) / max(min(share.values()), 1e-9)
print(f'IR ở cấp công ty xấp xỉ: {wa:.1f} (so với IR cấp mẫu ~1.66)')

In [ ]:
# 3. Phân bố lớp theo từng tập (chia theo thời gian, có dải purge)
for name, info in eda['splits'].items():
    total = info['total']
    dist = info['distress']
    print(f"{name:>10}: {dist:>3}/{total:<3} dương = {dist / total:.3f}")
hist = eda['history_length']
print('Số mẫu có < 5 quý lịch sử:', hist['n_below_min'], '/', hist['n_samples'])

## 4. Bảng so sánh mô hình và BASELINE (test n = 64)

In [ ]:
baselines = load('baselines.json')['rows']
header = f"{'hệ thống':<34} {'acc':>6} {'prec':>6} {'rec':>6} {'F1':>6} {'AUROC':>6} {'AP':>6}"
print(header)
print('-' * len(header))
for row in baselines:
    test = row.get('test') or {}
    print(f"{row['baseline']:<34} {test.get('accuracy', float('nan')):6.3f} "
          f"{test.get('precision', float('nan')):6.3f} {test.get('recall', float('nan')):6.3f} "
          f"{test.get('f1', float('nan')):6.3f} {test.get('auroc', float('nan')):6.3f} "
          f"{test.get('average_precision', float('nan')):6.3f}")
summary = load('summary.json')
print()
print('Mô hình được chốt:', summary['best_model'], '| ngưỡng:', round(summary['best_threshold'], 4))
print('Quy tắc chọn:', summary['selection_rule'])
for row in summary['models']:
    print(f"  {row['model']:<24} val AUROC {row['auroc']:.3f} | AP cross-company "
          f"{(row.get('cross_company_ap') or float('nan')):.4f}")

In [ ]:
# 5. Metric CHÍNH THỨC trên test tại đúng ngưỡng vận hành
te = load('test_evaluation.json')
tm = te['test_metrics']
op, half = tm['operating'], tm.get('confusion_at_0.5') or {}
print('mô hình:', te['model'], '| ngưỡng vận hành:', round(te['threshold'], 4))
print('tại ngưỡng vận hành:', {k: round(op[k], 3) for k in ('precision', 'recall', 'f1', 'macro_f1', 'mcc')})
print('confusion @ngưỡng vận hành:', {k: op.get(k) for k in ('tn', 'fp', 'fn', 'tp')})
print('confusion @0.5            :', {k: half.get(k) for k in ('tn', 'fp', 'fn', 'tp')})
print('số mẫu sai:', te['n_misclassified'])

In [ ]:
# 6. Ba mẫu sai (suy ra từ analysis.json — không nhập tay)
for case in load('analysis.json')['error_cases']:
    kind = 'FN (bỏ sót)' if case['actual'] == 1 else 'FP (báo động giả)'
    print(f"{case['sample_id']:<14} {kind:<16} P = {case['probability']:.4f} "
          f"| current_ratio = {case['current_ratio_latest']:.3f}")

## 7. Demo: chấm MỘT mẫu bằng hàm của repo (không viết lại logic)

In [ ]:
from scripts.predict import find_sample, format_report, load_model, score_sample

artifact = load_model()
sample, split = find_sample(sample_id='FIVE-2024Q3')
result = score_sample(sample, artifact, explain=True, top_k=4, n_coalitions=60, n_background=8)
print(format_report(result, split))

## 8. Kết luận nhanh

- Baseline `ticker_prior` (không dùng feature) đạt AUROC ≈ 0,986 — **không thua** mô hình ⇒ chỉ số
  in-domain không chứng minh năng lực dự báo; phải đọc kèm cross-company/LOCO.
- Mọi bảng ở trên đều đọc từ `reports/results/*.json`; chạy `python -m scripts.audit_data` để đối
  chiếu 98.185 phép kiểm tra (kỳ vọng 0 phát hiện).
- Bộ tài liệu đọc kèm: `docs/bo-tai-lieu-bao-ve.md`, `docs/checklist-doi-chieu-yeu-cau.md`,
  `docs/BAO-CAO.md` (mục 11 — tài liệu tham khảo).